# 조건을 바꿔 재실행하면 라벨이 얼마나 흔들리는가

2026-08-19에 배치 두 개가 **같은 요구사항 100건을 각각 라벨링했습니다.** 원래는 하나가
중간에 취소되고 잊힌 실행이었는데, 결과가 29일간 보관되는 덕에 뒤늦게 회수됐습니다.
덕분에 의도치 않게 **재실행 쌍**이 생겼습니다.

| | `batch_chunk2` | `batch_full_101_1024` |
|---|---|---|
| 제출 (UTC) | 2026-08-19 07:28 | 2026-08-19 12:04 |
| 앵커 풀 | **`anchor_pool_v3`** (192행) | **`anchor_pool_v2`** (100행) |
| 구조화 출력 | **`record_label` 도구 호출** | **`output_config.format`** |
| 응답 블록 | `tool_use` | `text` (+ 일부 `thinking`) |
| 결과 | 100건 중 86건 성공 (나머지는 취소) | 924건 |
| 쓰임 | 어디에도 안 쓰임 | **동결 라벨셋의 출처** |

모델(`claude-sonnet-5`), 프롬프트(`claude-rfp-risk-v5`), 인출 전략, 실행 경로(배치),
입력 파일은 같습니다. 그런데 **다른 조건이 하나가 아니라 둘입니다.** 두 실행 사이인
같은 날 저녁 커밋 `3c0dba9`(“use structured outputs in batch pipeline”)가 배치
파이프라인을 도구 호출에서 json_schema 구조화 출력로 바꿨습니다. 두 배치의 원본 응답을
API에서 직접 받아 블록 구성으로 확인한 사실입니다.

**그래서 이 쌍은 앵커 풀 효과를 분리하는 실험이 아닙니다.** 조건 두 개와 재실행 변동이
함께 섞여 있고, 아래 숫자는 그 셋을 합친 총량입니다.

### 용어

- **앵커(anchor)**: 프롬프트에 정답 예시로 함께 싣는, 이미 라벨된 요구사항입니다.
  few-shot 학습의 "예시"에 해당합니다.
- **인출(retrieval)**: 요구사항 하나를 라벨링할 때마다 앵커 풀에서 3건을 골라 넣는
  과정입니다. 타깃이 달라지면 뽑히는 앵커도 달라집니다.
- **자카드 지수(Jaccard)**: 두 집합이 얼마나 겹치는지를 0~1로 잰 값입니다.
  `교집합 / 합집합`이고, 1이면 완전히 같은 앵커를 봤다는 뜻입니다.

### 무엇을 왜 재는가

라벨을 LLM이 만들었으니 "이 라벨이 맞느냐"를 직접 물을 수는 없습니다. 대신 **조건을
바꿨을 때 같은 답이 나오는가**는 물을 수 있고, 그 흔들림이 라벨 품질의 상한입니다.

두 실행의 차이에는 세 가지가 섞여 있습니다 — 앵커 풀이 만든 차이, 구조화 출력 방식이
만든 차이, 그리고 아무것도 안 바꿔도 나오는 재실행 변동입니다. LLM은 같은 입력에도
매번 똑같이 답하지 않기 때문입니다. 조건별 실행이 각 1회뿐이라 이 셋을 가를 정보가
없습니다.

그래서 관측값을 결정 23이 같은 조건 반복에서 잰 **37/40(92.5%)** 과 나란히 놓고,
**Wilson 95% 신뢰구간이 겹치는지**를 봅니다. 겹치면 관측된 차이를 어떤 조건 탓으로도
돌릴 수 없습니다. 결정 34가 골드 11건을 우열 판정에서 물린 것과 같은 검사입니다.

### 이 노트북이 바꾸지 않는 것

데이터셋은 동결입니다. 이 노트북은 라벨을 고치지도, 앵커 풀을 고치지도 않습니다.
**측정 기록**이고, 008·010이 적어둔 한계에 숫자를 하나 더하는 것이 전부입니다.

주의할 한계가 하나 더 있습니다 — 공통 86건은 **전부 `defense_intelligent_platform`
한 문서**입니다. 문서마다 라벨 분포가 다르므로 이 수치를 전체로 일반화하면 안 됩니다.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / 'scripts').is_dir()), Path.cwd().resolve())
sys.path.insert(0, str(ROOT))

from scripts.evaluation.run_agreement import (agreement_rows, compare_runs, confusion_rows,
                                              load_run_results, repeat_baseline_agreement,
                                              same_document_anchor_count,
                                              stratify_by_anchor_overlap)

pool_v3 = load_run_results(ROOT / 'reports/current/claude_batches/batch_chunk2/results.jsonl')
pool_v2 = load_run_results(ROOT / 'reports/current/claude_runs/batch_full_101_1024/results.jsonl')
comparison = compare_runs(pool_v3, pool_v2, left_name='v3풀', right_name='v2풀')

# 대조가 성립하는지 먼저 확인한다. 결정 10(자기 문서 앵커 금지)이 어느 한쪽에서라도
# 깨졌다면 두 실행은 조건이 다른 게 아니라 한쪽이 오염된 것이라 비교가 무의미하다.
print(f'공통 요구사항 {len(comparison.uids)}건 · 문서 {sorted({u.split(":")[0] for u in comparison.uids})}')
print(f'자기 문서 앵커 주입 — v3풀 {same_document_anchor_count(pool_v3)}건 · '
      f'v2풀 {same_document_anchor_count(pool_v2)}건  (결정 10은 0을 요구한다)')

identical = comparison.identical_anchor_uids
print(f'두 실행이 완전히 같은 앵커를 본 건: {len(identical)}/{len(comparison.uids)}')

# 필드별 일치율. 주 라벨만 보면 안정성을 과대평가하므로 보조 축도 함께 본다.
display(pd.DataFrame(agreement_rows(comparison.agreements.values())).set_index('항목').style.format(
    {'일치율': '{:.1%}', 'Wilson 95% 하한': '{:.3f}', 'Wilson 95% 상한': '{:.3f}'}))


In [ ]:
# 어느 방향으로 갈렸는가. 행이 v3풀, 열이 v2풀이다.
display(pd.DataFrame(confusion_rows(comparison)).set_index('').rename_axis('v3풀 → v2풀', axis=1))

# 갈린 건들의 앵커 자카드. 1.00인데도 갈렸다면 앵커로는 설명되지 않는다.
display(pd.DataFrame([{'요구사항': d.requirement_uid.split(':')[1], 'v3풀': d.left,
                       'v2풀': d.right, '앵커 자카드': d.anchor_jaccard}
                      for d in comparison.disagreements]
                     ).set_index('요구사항').style.format({'앵커 자카드': '{:.2f}'}))

# 핵심 검사 — 앵커 중복도로 층을 갈랐을 때 일치율이 올라가는가,
# 그리고 그 차이가 동일 조건 반복 변동과 구별되는가.
baseline = repeat_baseline_agreement()
measured = [*stratify_by_anchor_overlap(comparison, pool_v3, pool_v2), comparison.primary, baseline]
table = pd.DataFrame(agreement_rows(measured)).set_index('항목')
table['기준선과 구간 겹침'] = [a.overlaps(baseline) for a in measured]
display(table.style.format({'일치율': '{:.1%}', 'Wilson 95% 하한': '{:.3f}',
                            'Wilson 95% 상한': '{:.3f}'}))


## 결론

**주 라벨 일치율은 74/86 = 86.0%, Wilson 95% CI [0.772, 0.918]입니다.** 결정 23의
동일 조건 반복 기준선 37/40 = 92.5%, CI [0.801, 0.974]와 **구간이 겹칩니다.**

그래서 이 표본에서 나온 답은 이것입니다 — **두 실행의 차이를 어느 조건 탓으로도 돌릴
수 없습니다.** 14%의 불일치는 아무것도 안 바꿔도 나올 수 있는 크기입니다. "조건들의
효과가 없다"가 아니라 **"n=86으로는 재실행 변동과 구별되지 않는다"** 입니다. 둘은 다른
말이고, 후자가 지금 말할 수 있는 전부입니다.

앵커 중복도로 층을 갈라도 같습니다. 앵커 동일 10/11(90.9%) → 부분 겹침 53/62(85.5%)
→ 앵커 상이 11/13(84.6%)로 **방향은 예상대로**지만 세 구간이 거의 완전히 겹칩니다.
앵커가 완전히 같았던 11건 중에도 1건(`SFR-022`)이 갈렸습니다. 그리고 이 층화는 앵커
축만 가른 것이라 **출력 방식 차이는 세 층에 그대로 남아 있습니다** — 앵커 단독 효과의
근거로 쓸 수 없는 또 하나의 이유입니다.

### 이미 기록된 것에 대한 보정

`docs/issues/010`과 `decisions-05.md`는 앵커 풀 라벨과 동결 라벨의 16% 불일치를
두고 **"결정 23의 반복 변동 7.5%의 약 2배"** 라고 적었습니다. 그 문장은 CI 검사를
통과하지 못합니다. 84/100의 CI는 [0.756, 0.899]로 37/40의 [0.801, 0.974]와 겹칩니다.
같은 문서가 골드 11건과 37/40 대 36/40에는 이미 CI 겹침 논리를 적용했으니, **16% 대
7.5%에만 적용하지 않은 것은 일관성 문제**입니다. 16%라는 관측 자체는 유효하고,
"2배"라는 해석만 근거가 약합니다.

### 그 밖에 확인된 것

- **결정 10은 지켜졌습니다.** 자기 문서 앵커 주입이 양쪽 실행 모두 0건입니다.
- **불안정한 경계는 `견적반영` ↔ `계약·질의검토`입니다.** 갈린 12건 중 9건이 이
  경계이고, `통상수용`으로 판정된 20건은 20/20 완전 일치였습니다. 보조 축도 같은
  방향입니다 — `build_difficulty` 91.9%는 안정적인데 `cost_basis` 72.1%,
  `domain_dependency` 69.8%는 주 라벨보다 훨씬 많이 흔들립니다. **보조 축을 근거로
  쓰는 주장은 주 라벨보다 약하게 잡아야 합니다.**
- **004(실행 경로 교란)는 배치 안에서도 생깁니다.** 004는 동기 대 배치를 봤지만, 같은
  배치 경로라도 구조화 출력 방식이 바뀌면 생성 경로가 달라집니다. 실제로 도구 호출을
  강제한 `chunk2` 응답에는 `thinking` 블록이 하나도 없고, json_schema 쪽에는 있습니다.
  실행 조건을 기록할 때 "동기/배치"라는 경로 이름만으로는 부족하다는 뜻입니다.
- **`ECR-029`가 흥미롭습니다.** v3풀 실행은 `계약·질의검토`를 직접 내놨는데, 이는
  결정 30의 고정 규칙이 나중에 v2풀 실행의 `견적반영`을 보정한 값과 같습니다.
  규칙이 임의로 덮어쓴 것이 아니라 모델이 다른 조건에서 스스로 도달하는 값이라는
  방증입니다(n=1이므로 그 이상은 아닙니다).
- **동결 라벨셋은 이 실행과 정합적입니다.** v2풀 실행과 `label_dataset_v3`의
  불일치는 86건 중 2건뿐이고, 둘 다(`ECR-006`·`ECR-029`) 매니페스트에 기록된 규칙
  보정입니다. 즉 동결 데이터셋은 원 실행을 충실히 반영하고 있습니다.
  `label_dataset_v4`(2026-08-28)는 v3에 `normalized_requirement_text`·`model_text`를
  더한 것이고 **`primary_action`은 1,024건 전부 v3와 같으므로**, 이 결론은 현재
  데이터셋에도 그대로 적용됩니다.

### 남는 작업

없습니다. 데이터셋·앵커 풀 모두 동결이고 이 측정은 그중 무엇도 바꾸지 않습니다.
논문에는 한 문장이면 충분합니다 — 조건을 바꾼 재실행의 라벨 일치율은 86.0%였고,
동일 조건 반복 변동과 통계적으로 구별되지 않았다.
